Generate a chart for a validation song and compare it with the ground truth

Runs top to bottom: setup, load validation data, load model, generate, export.

In [6]:
import sys, json, warnings
import torch
from pathlib import Path
from hydra import initialize_config_dir, compose

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

CONFIG_DIR = PROJECT_ROOT / "conf"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with initialize_config_dir(version_base=None, config_dir=str(CONFIG_DIR)):
    cfg = compose(config_name="train")

experiment_config = cfg.experiment
hyperparams = cfg.hyperparams
segment_frames = hyperparams["segment_frames"]
batch_size = hyperparams["batch_size"]

warnings.filterwarnings("ignore", message=".*nested tensors.*")

In [ ]:
# NOTE: dataset import path is from the old notebook; adjust it if the dataset moved into `tracers`.
from tracers.datasets import AutoOsu4KDataset, split_song_ids, make_dataloader

ROOT = PROJECT_ROOT / "dataset"
AUDIO_DIR = ROOT / "audio"
BEATMAP_DIR = ROOT / "beatmap" / "4keys"

# The base dataset is only used to reproduce the same song split as training.
base_dataset = AutoOsu4KDataset(
    beatmap_dir=BEATMAP_DIR,
    audio_dir=AUDIO_DIR,
    segment_frames=segment_frames,
    crop_mode="random",
    audio_cache_size=8,
)
_, val_ids, _ = split_song_ids(
    base_dataset.song_ids,
    val_fraction=0.10,
    test_fraction=0.10,
    seed=42,
)


In [8]:
from tracers.models.transformer import TracerNpModel
from tracers.training.trainer import Trainer
from tracers.generate.converter import predictions_to_json, export_generation_debug

model = TracerNpModel(hyperparams).to(device)

# The optimizer is only here because load_checkpoint() also restores its state.
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=hyperparams["learning_rate"],
    weight_decay=hyperparams["weight_decay"],
)

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    train_loader=None,
    valid_loader=None,
    device=device,
    experiment_config=experiment_config,
    hyperparams=hyperparams,
)

CHECKPOINT = PROJECT_ROOT / "notebooks/checkpoints/best.pt"
trainer.load_checkpoint(CHECKPOINT)

Resumed from epoch 76. Best validation loss: 0.1494


In [ ]:
context_frames = 500

val_dataset = AutoOsu4KDataset(
    beatmap_dir=BEATMAP_DIR,
    audio_dir=AUDIO_DIR,
    segment_frames=segment_frames,
    crop_mode="all",
    stride_frames=segment_frames - context_frames,
    song_ids=val_ids,
    audio_cache_size=8,
)

val_loader = make_dataloader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0,
)

In [ ]:
# ---------- settings ----------
CHART_ID = None          # e.g. "1055661-148-4"; None = first chart in the validation set
BPM = 120                # only used to place notes in the exported JSON
OFFSET_MS = 0
LANE_MAP = (0, 2, 4, 6)

TEMPERATURE = None        # None = greedy
TOP_K = None
TOP_P = None
SEED = 0
SAVE_DEBUG_CSV = True

# ---------- collect the chart's segments ----------
INPUT_KEYS = ["specs", "beat_phase", "beat_num", "difficulty", "valid_frames"]
segments, found = [], False

for batch in val_loader:
    for i, cid in enumerate(batch["chart_id"]):
        if CHART_ID is None:
            CHART_ID = cid
        if cid != CHART_ID:
            if found:
                break
            continue
        found = True
        segments.append({
            "start": int(batch["start_frame"][i]),
            "sample": {k: batch[k][i:i+1] for k in INPUT_KEYS},
            "actual": batch["actions_target"][i].cpu(),
        })
    else:
        continue
    break

segments.sort(key=lambda s: s["start"])
print(f"Chart {CHART_ID}: {len(segments)} segments")

Chart 1055661-148-4: 8 segments


In [ ]:
from tracers.generate.converter import generate_validation_chart

generated, actual, debug_segments = generate_validation_chart(
    trainer,
    val_loader,
    chart_id=CHART_ID,
    context_frames=context_frames,
    seed_action=None,  # Use ground-truth initial action
    temperature=TEMPERATURE,
    top_k=TOP_K,
    top_p=TOP_P,
    seed=SEED,
    collect_debug=SAVE_DEBUG_CSV,
)

Frame      0 | frames 2000 | active   23 (actual   68) | next seed 0
Frame   2000 | frames 2000 | active    8 (actual   56) | next seed 0
Frame   4000 | frames 2000 | active   17 (actual   50) | next seed 0
Frame   6000 | frames 2000 | active   36 (actual   50) | next seed 0
Frame   8000 | frames 2000 | active   52 (actual   64) | next seed 0
Frame  10000 | frames 2000 | active   61 (actual   63) | next seed 0
Frame  12000 | frames 2000 | active   40 (actual   57) | next seed 0
Frame  14000 | frames 946 | active    0 (actual   12) | next seed 0

Done. 14946 frames (149.5 s)
Active frames: generated 237 | actual 420


In [ ]:
first_start = segments[0]["start"]
valid_all = torch.ones(len(generated), dtype=torch.bool)

def to_events(actions):
    return predictions_to_json(
        action_predictions=actions,
        bpm=BPM,
        start_frame=first_start,
        offset_ms=OFFSET_MS,
        lane_map=LANE_MAP,
        valid_frames=valid_all,
    )

output_dir = Path(f"charts/{CHART_ID}")
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / "generated_np_chart.json", "w", encoding="utf-8") as f:
    json.dump(to_events(generated), f, indent=4)

with open(output_dir / "actual_full_chart.json", "w", encoding="utf-8") as f:
    json.dump(to_events(actual), f, indent=4)

print("Saved generated_np_chart.json and actual_full_chart.json")

if SAVE_DEBUG_CSV:
    export_generation_debug(debug_segments, output_path=str(output_dir / "generation_np_debug.csv"), top_k=5)

Saved generated_np_chart.json and actual_full_chart.json
Exported 14946 frame records to: charts\1055661-148-4\generation_np_debug.csv
